# Use case — `Utility/time_delay_pspline.py`

Adaptive shared P-spline estimation for two, three or four components. The first component is the zero-delay reference.

**Convention:** `t_shifted_k = t_k - delay_k`. Inspect LOO, overlap and K profiles before interpreting the selected delay.

In [ ]:
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name.lower() == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent
sys.path.insert(0, str(PROJECT_ROOT)) if str(PROJECT_ROOT) not in sys.path else None

from Utility import time_delay_pspline as td

In [ ]:
INPUT_CSV = PROJECT_ROOT / "data" / "cleaned_lightcurves.csv"  # Canonical preprocessed light curves.
SOURCE_ID = "Source_ID"                               # System containing all requested components.
COMPONENT_IDS = [                                                # Two to four IDs; first entry defines delay zero.
    "Component_A_ID",
    "Component_B_ID",
]
COMPONENT_NAMES = ["A reference", "B"]                         # Human-readable labels matching COMPONENT_IDS.

if not INPUT_CSV.exists():
    raise FileNotFoundError(f"Supply the cleaned canonical CSV first: {INPUT_CSV}")

## Estimator hyperparameters

This complete dictionary preserves the original `TDPSPL.ipynb` adaptation. For a first run, replace it with `td.load_parameter_profile("quick")`.

In [ ]:
# ============================================================
# CONFIGURATION DE L'ESTIMATION DU RETARD TEMPOREL PAR P-SPLINE
# ============================================================
#
# Pour une première utilisation, il est conseillé de conserver
# ces valeurs. Les paramètres les plus souvent modifiés sont :
# dmin, dmax, ngrid, min_points, min_frac et min_span_frac.
#
# La première courbe donnée au programme est toujours la référence.
# Son retard est fixé à 0 jour. Les retards des autres courbes sont
# calculés par rapport à cette première courbe.


estimator_kwargs = {

    # --------------------------------------------------------
    # 1. INTERVALLE DANS LEQUEL LE RETARD EST RECHERCHÉ
    # --------------------------------------------------------

    # Plus petit retard que le programme est autorisé à tester.
    # Ici, la recherche commence à -500 jours.
    # Diminuer cette valeur, par exemple à -700, pour chercher
    # des retards négatifs plus grands.
    "dmin": -500,

    # Plus grand retard que le programme est autorisé à tester.
    # Ici, la recherche s'arrête à +500 jours.
    # Augmenter cette valeur si un retard supérieur à 500 jours
    # est scientifiquement possible.
    "dmax": 500,

    # Nombre de retards testés entre dmin et dmax.
    # Une valeur plus grande donne une recherche initiale plus précise,
    # mais augmente fortement le temps de calcul.
    # Avec -500, +500 et 400 valeurs, l'écart est d'environ 2.5 jours.
    "ngrid": 400,

    # Nombre de fois où le programme réoptimise successivement
    # les retards de toutes les composantes.
    # Une passe suffit généralement pour deux courbes.
    # Pour trois ou quatre courbes, utiliser 2 ou 3 peut être plus robuste,
    # mais cela multiplie le temps de calcul.
    "max_delay_passes": 1,

    # Précision numérique du raffinement final, en jours.
    # 0.05 jour correspond à environ 1.2 heure.
    # Attention : ce n'est pas l'incertitude scientifique du résultat.
    "scalar_xatol": 0.05,


    # --------------------------------------------------------
    # 2. FORME DE LA COURBE P-SPLINE
    # --------------------------------------------------------

    # Degré des morceaux polynomiaux de la spline.
    # 3 correspond à une spline cubique, qui est le choix standard.
    # Il est généralement déconseillé de modifier ce paramètre.
    "degree": 3,

    # Ancien paramètre utilisé pour choisir automatiquement le nombre
    # de nœuds de la spline.
    # Avec K_selection="loo", il a normalement très peu d'effet.
    # La valeur 999 force généralement l'ancienne règle à proposer K_min.
    # Il est principalement conservé comme solution de secours.
    "points_per_interval": 999,

    # Plus petit nombre de nœuds internes que le programme peut tester.
    # Une petite valeur produit une courbe plus simple et plus lisse.
    "K_min": 5,

    # Plus grand nombre de nœuds internes que le programme peut tester.
    # Une grande valeur autorise une courbe plus détaillée, mais augmente
    # le risque de suivre le bruit et augmente le temps de calcul.
    # Le programme peut automatiquement utiliser une limite plus petite
    # si le nombre d'observations est insuffisant.
    "K_max": 150,


    # --------------------------------------------------------
    # 3. CHOIX AUTOMATIQUE DU NOMBRE DE NŒUDS K
    # --------------------------------------------------------

    # Méthode utilisée pour choisir la complexité de la spline.
    # "loo" teste la capacité de la spline à prédire chaque observation
    # lorsqu'elle est temporairement retirée du modèle.
    # C'est la méthode recommandée.
    "K_selection": "loo",

    # Liste exacte des valeurs de K à tester.
    # None demande au programme de construire automatiquement cette liste.
    #
    # Exemple de liste manuelle :
    # "K_loo_candidates": [5, 10, 20, 40, 80]
    #
    # Pour une utilisation normale, laisser None.
    "K_loo_candidates": None,

    # Nombre approximatif de valeurs de K testées lorsque
    # K_loo_candidates=None.
    # Augmenter cette valeur explore plus de modèles, mais ralentit
    # considérablement le calcul.
    "K_loo_n_candidates": 15,

    # Pénalité ajoutée aux splines trop complexes lors du choix de K.
    # Une valeur plus grande favorise des splines plus simples.
    # Une valeur plus petite autorise des splines plus flexibles.
    "K_loo_complexity_penalty": 1.0,

    # Limite de sécurité contre les modèles extrêmement flexibles.
    # Un modèle est rejeté si sa complexité effective dépasse 80 %
    # du nombre d'observations disponibles dans le recouvrement.
    # Il est conseillé de conserver cette valeur.
    "K_loo_df_cap_frac": 0.80,


    # --------------------------------------------------------
    # 4. CRITÈRE UTILISÉ POUR CHOISIR LE RETARD
    # --------------------------------------------------------

    # Score que le programme minimise pour trouver le retard.
    #
    # "loo"  : qualité de prédiction Leave-One-Out, recommandée.
    # "bic"  : compromis entre qualité d'ajustement et complexité.
    # "chi2" : qualité d'ajustement directe, plus sensible au surajustement.
    #
    # Pour l'analyse principale, conserver "loo".
    "delay_score": "loo",


    # --------------------------------------------------------
    # 5. ESTIMATION LOCALE DU BRUIT ET DE LA VARIABILITÉ
    # --------------------------------------------------------

    # Taille demandée pour le voisinage utilisé afin d'estimer
    # localement la dispersion des flux.
    #
    # Important : dans la version actuelle, le programme cherche
    # généralement à utiliser au moins 5 points. Une valeur de 3
    # conduit donc souvent à un voisinage réel proche de 5 points.
    "rolling_window": 3,

    # Manière de construire le voisinage local.
    #
    # "points" : utilise les observations voisines, même si elles sont
    #            éloignées dans le temps.
    # "time"   : utilise toutes les observations situées dans un rayon
    #            temporel donné.
    #
    # "points" est simple et fonctionne même avec un échantillonnage
    # temporel irrégulier.
    "rolling_mode": "points",

    # Rayon temporel, en jours, utilisé uniquement lorsque
    # rolling_mode="time".
    #
    # None demande au programme de l'estimer automatiquement à partir
    # de l'espacement habituel entre les observations.
    # Ce paramètre est ignoré avec rolling_mode="points".
    "rolling_time_radius": None,

    # True combine l'erreur de mesure fournie dans les données avec
    # la dispersion locale observée autour de chaque date.
    # Cela évite de donner trop d'importance aux zones très variables.
    #
    # False utilise seulement les erreurs de mesure.
    "use_global_mad_in_sigma": True,

    # Valeur minimale autorisée pour les erreurs et les dispersions
    # après normalisation des courbes.
    # Elle empêche une observation ayant une erreur presque nulle
    # de dominer tout le calcul.
    "mad_floor": 0.05,


    # --------------------------------------------------------
    # 6. INTENSITÉ DU LISSAGE DE LA P-SPLINE
    # --------------------------------------------------------

    # Niveau général de lissage.
    #
    # "auto" demande au programme de le calculer automatiquement
    # à partir de la dispersion observée dans les courbes.
    #
    # Il est également possible d'imposer une valeur numérique,
    # par exemple 1.0, mais "auto" est recommandé.
    "lambda_base": "auto",

    # Facteur appliqué au niveau de lissage automatique.
    #
    # Augmenter cette valeur produit une courbe plus lisse.
    # Diminuer cette valeur produit une courbe plus flexible.
    #
    # Ce paramètre est utilisé seulement si lambda_base="auto".
    "lambda_scale": 0.03,

    # Contrôle la variation du lissage selon les régions temporelles.
    #
    # 0.0 : même lissage partout.
    # 0.5 : adaptation modérée aux variations locales.
    # 1.0 : adaptation plus forte.
    #
    # Dans ce code, une forte dispersion locale entraîne un lissage
    # local plus important.
    "lambda_alpha": 0.5,

    # Plus petite pénalisation locale autorisée, exprimée comme
    # une fraction du niveau général lambda_base.
    #
    # 0.05 signifie qu'une région peut être au maximum 20 fois
    # moins lissée que le niveau général.
    "lambda_min_ratio": 0.05,

    # Plus grande pénalisation locale autorisée.
    #
    # 5.0 signifie qu'une région peut être au maximum 5 fois
    # plus lissée que le niveau général.
    "lambda_max_ratio": 5.0,


    # --------------------------------------------------------
    # 7. DIAGNOSTICS BIC
    # --------------------------------------------------------
    #
    # Avec delay_score="loo", les deux paramètres suivants ne changent
    # pas directement le retard final. Ils servent principalement
    # au calcul du diagnostic BIC affiché dans les résultats.

    # Limite de complexité utilisée par le diagnostic BIC.
    # Une forte pénalité est appliquée si la complexité effective
    # dépasse 30 % du nombre d'observations.
    "df_cap_frac": 0.30,

    # Importance donnée à la courbure de la spline dans le score BIC.
    # Une valeur plus grande pénalise davantage les courbes irrégulières.
    # Avec delay_score="loo", conserver simplement 1.0.
    "rough_penalty": 1.0,


    # --------------------------------------------------------
    # 8. RECOUVREMENT TEMPOREL ENTRE LES COURBES
    # --------------------------------------------------------

    # Pénalité supplémentaire lorsque le décalage temporel fait perdre
    # une partie importante des observations.
    #
    # 0.0 désactive cette pénalité douce.
    # Les conditions obligatoires min_points, min_frac et min_span_frac
    # restent actives même lorsque cette valeur vaut zéro.
    "overlap_penalty": 0.0,

    # Durée de recouvrement idéale utilisée par la pénalité douce.
    # 1.0 représente 100 % de la durée de la courbe la plus courte.
    # Ce paramètre n'a aucun effet lorsque overlap_penalty=0.0.
    "min_span_frac_ref": 1.0,

    # Fraction idéale d'observations conservées dans chaque composante.
    # 1.0 représente 100 % des observations.
    # Ce paramètre n'a aucun effet lorsque overlap_penalty=0.0.
    "min_frac_ref": 1.0,

    # Nombre minimal d'observations que chaque composante doit conserver
    # dans la partie temporelle commune.
    # Si une seule composante conserve moins de 10 points, le retard
    # testé est automatiquement rejeté.
    "min_points": 10,

    # Fraction minimale des observations conservées dans chaque courbe.
    # 0.50 signifie qu'au moins 50 % des points de chaque composante
    # doivent appartenir au recouvrement temporel.
    "min_frac": 0.50,

    # Fraction minimale de la durée temporelle conservée.
    # 0.50 signifie que le recouvrement doit couvrir au moins 50 %
    # de la durée de la courbe la plus courte.
    #
    # Pour des courbes couvrant 10 ans, cela correspond approximativement
    # à un minimum de 5 ans de recouvrement.
    "min_span_frac": 0.50,


    # --------------------------------------------------------
    # 9. CHOIX ET AFFICHAGE DU RÉSULTAT FINAL
    # --------------------------------------------------------

    # Pour deux courbes et delay_score="loo", True impose que le retard
    # final corresponde exactement au plus petit score LOO enregistré
    # et visible sur le graphique.
    #
    # Ce paramètre est ignoré avec trois ou quatre courbes.
    # Si overlap_penalty devient supérieur à zéro, il peut être préférable
    # de mettre False afin de respecter le score total pénalisé.
    "force_final_to_min_plotted_loo": True,

    # True affiche la progression, le retard trouvé, le score,
    # le nombre de nœuds, la complexité et le recouvrement.
    #
    # False effectue les mêmes calculs sans afficher les détails.
    "verbose": True,
}

MC_SAMPLES = 300                 # Flux-error draws; use 20 for a smoke test and >=300 for final work.
MC_RANDOM_SEED = 42              # Reproducible random-number seed.
MC_ERROR_SCALE = 1.0             # 1.0 uses flux_obs_error exactly.
MC_FORCE_SAME_K = True           # Reuse base-fit K; False propagates K selection but is much slower.
MC_PROGRESS_EVERY = 10           # Print progress every N draws.

In [ ]:
df = td.load_lightcurve_csv(INPUT_CSV)
system = td.get_components_from_df(
    df=df,
    source_id=SOURCE_ID,
    comp_ids=COMPONENT_IDS,
    names=COMPONENT_NAMES,
)
result = td.estimate_time_delay_pspline(
    system["curves"],
    **estimator_kwargs,
)
td.print_result_multi(system, result)
display(result["pair_delays"])

## Required diagnostics

A delay is not accepted from its scalar value alone. Inspect the objective, LOO, overlap, selected K, fitted curves, rolling-MAD sigma and residuals.

In [ ]:
td.plot_delay_profiles_multi(result, y_col="cost")
td.plot_LOO_score_profile(result)
td.plot_overlap_profile(result)
td.plot_K_profile(result)
td.plot_K_loo_table(result)
td.plot_fit_multi(result, degree=estimator_kwargs["degree"], show_knots=True)
td.plot_global_mad_sigma_diagnostics(result)
td.plot_residuals_multi(result)

## Measurement-error uncertainty

Every draw perturbs each flux by its own `flux_obs_error`, then reruns the estimator. The sample can be multimodal; always inspect its histogram and saved draws.

In [ ]:
uncertainty = td.run_fluxobs_error_mc_pspline(
    system=system,
    estimator_kwargs=estimator_kwargs,
    n_samples=MC_SAMPLES,
    random_seed=MC_RANDOM_SEED,
    error_scale=MC_ERROR_SCALE,
    base_res=result,
    force_same_K_as_base=MC_FORCE_SAME_K,
    progress_every=MC_PROGRESS_EVERY,
    verbose=True,
)
td.print_fluxobs_error_mcmc_uncertainty(uncertainty)
td.plot_fluxobs_error_mcmc_uncertainty(uncertainty, bins=30)
display(uncertainty["pair_delay_summary"])

## Batch command

```bash
python -m Utility.time_delay_pspline data/cleaned_lightcurves.csv --pairs configs/time_delay_system_pairs.csv --profile legacy_notebook --mc-samples 300 --output-dir results/batch_time_delays
```